# Basic

Same analysis as the hop-based notebook 5, on `metrics_results_prim_fd/` (status-dependent decay for selection,
primitive-hop distance score for evaluation). Figures are written to `decay_with_prim_fd/figures/`.

In [ ]:
%load_ext autoreload
%autoreload all

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, where src/ lives

import config_prim_fd as config             # found in decay_with_prim_fd/
import prim_fd_tokenizer as pf              # status-dependent coverage, selection and distance score

import os
import polars as pl
import numpy as np
import networkx as nx  
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


import src.graph_tokenizer_gd_tree_dev.graph_fct as graph_fct

df_relations = pl.read_parquet(f"{config.BasicConfig().relation_path}")
df_mapped = pl.read_parquet(f"{config.BasicConfig().mapped_path}")
mapped_ids = df_mapped["id"].unique().to_list()

BASELINE_FAMILY = {'closeness_centrality': 'centrality-based',
 'discrete_set_cover': 'ours(ablation)',
 'highest_degree': 'degree-based',
 'highest_degree_dist_1': 'degree-based',
 'k_random_all_samples': 'random',
 'most_children': 'degree-based',
 'pagerank': 'centrality-based',
 'personalized_pagerank': 'centrality-based',
 '0.0': 'ours',
 '0.1': 'ours',
 '0.2': 'ours',
 '0.3': 'ours',
 '0.4': 'ours',
 '0.5': 'ours',
 '0.6': 'ours',
 '0.7': 'ours',
 '0.8': 'ours',
 '0.9': 'ours',
 '1.0': 'ours'
 } 

perf_df_all = (pl.read_parquet(config.Results().perf_all)
                .rename({"conciseness": "Tokens_per_concept"})
               .with_columns(pl.col("file_type").replace(BASELINE_FAMILY).alias("family"))
               .drop("exact_rate"))

METRIC_CORE = ['Tokens_per_concept', 'distance_score', 'unique_rate', 'unknown_rate', 'semantic_coverage']

os.makedirs("figures", exist_ok=True)   # figures of this variant go to decay_with_prim_fd/figures/

# 1. subgraph construction and comparison

In [ ]:
# original graph
G_original = graph_fct.build_relations_graph(df_relations, col_src="src.id", col_dst="dst.id", col_relation="relation")
print(f"Original graph: {G_original.number_of_nodes()} nodes, {G_original.number_of_edges()} edges, number of connected components: {nx.number_weakly_connected_components(G_original)},number of distinct relation types: {len(set(nx.get_edge_attributes(G_original, 'relation').values()))}")

In [ ]:
# computed subgraph 
id_to_label, G_d = graph_fct.get_combined_combined_subgraphs_and_id2label()
print(f"Computed subgraph: {G_d.number_of_nodes()} nodes, {G_d.number_of_edges()} edges, number of connected components: {nx.number_weakly_connected_components(G_d)},number of distinct relation types: {len(set(nx.get_edge_attributes(G_d, 'relation').values()))}")

In [ ]:
nx.is_directed_acyclic_graph(G_d),nx.is_directed_acyclic_graph(G_original),

In [ ]:
# omited relation types in the computed subgraph
omited_relation_types = set(nx.get_edge_attributes(G_original, 'relation').values()) - set(nx.get_edge_attributes(G_d, 'relation').values())
omited_relation_types

# rank best methods

In [ ]:
compression_level_dict = {
    "high": [500, 6500],      # small k, most compression
    "mid":  [7000, 13000],
    "low":  [13500, 19500],   # large k, least compression
}


higher_is_better = {
    "semantic_coverage": True,
    "distance_score": True,
    "unique_rate": True,
    "Tokens_per_concept": False,
    "unknown_rate": False,
}


# compression rate for each compression level
for compression_level, (k_min, k_max) in compression_level_dict.items():
    print(f"Compression level: {compression_level}, k_min: {k_min}, k_max: {k_max}")
    print(f"compression rate: {k_min/len(mapped_ids):.2%} - {k_max/len(mapped_ids):.2%}")

# rank all methods by each metric, and compute the average rank across all metrics

In [ ]:
# (perf_df_all
#  .filter(pl.col("family") != pl.col("file_type"))
# .select(["category", "file_type", "k", "family"] + METRIC_CORE)
# .with_columns(
#         pl.when(pl.col("k").is_between(*compression_level_dict["high"]))
#           .then(pl.lit("high"))
#         .when(pl.col("k").is_between(*compression_level_dict["mid"]))
#           .then(pl.lit("mid"))
#         .when(pl.col("k").is_between(*compression_level_dict["low"]))
#           .then(pl.lit("low"))
#         .otherwise(None)
#         .alias("compression_level"))

# .with_columns([
#         pl.col(m)
#           .rank(method="average", descending=higher_is_better[m])
#           .over(["family", "compression_level"])
#           .alias(f"{m}_rank")
#         for m in METRIC_CORE
#     ])
# .with_columns(pl.mean_horizontal([f"{m}_rank" for m in METRIC_CORE]).alias("avg_rank"))
#  .group_by(["compression_level", "family", "file_type"])
#  .agg(pl.col("avg_rank").mean().alias("avg_rank_over_k"))
#  .sort(["compression_level", "avg_rank_over_k"],)

# )

In [ ]:
perf_df_all.filter(pl.col("family") != pl.col("file_type"))

In [ ]:
df_rank_input = (
    perf_df_all
    .filter(pl.col("family") != pl.col("file_type"))
    # 2) random: average the 20 draws -> one row per (method, k)
    .group_by(["family", "file_type", "k"])
    .agg(METRIC_CORE)
    .with_columns(
        pl.when(pl.col("k").is_between(*compression_level_dict["high"])).then(pl.lit("high"))
          .when(pl.col("k").is_between(*compression_level_dict["mid"])).then(pl.lit("mid"))
          .when(pl.col("k").is_between(*compression_level_dict["low"])).then(pl.lit("low"))
          .alias("compression_level"))
)

ranked = (
    df_rank_input
    # 3) rank METHODS against each other at the same k (rank 1 = best)
    .with_columns([
        pl.col(m).rank(method="average", descending=higher_is_better[m])
          .over("k").alias(f"{m}_rank")
        for m in METRIC_CORE])
    .with_columns(pl.mean_horizontal([f"{m}_rank" for m in METRIC_CORE]).alias("avg_rank"))
)

rank_table = (
    ranked
    # 4) average over the 13 k values of each level
    .group_by(["compression_level", "family", "file_type"])
    .agg([pl.col(f"{m}_rank").mean() for m in METRIC_CORE]
         + [pl.col("avg_rank").mean()])
    .sort(["compression_level", "avg_rank"])
)
ranked
rank_table.sort("avg_rank")
rank_table.write_csv(config.Results().perf_ranked_csv)

In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D

# rank_df = your rank_table (polars); here read from the exported CSV
rank_df = rank_table

LEVELS = [("high", "High compression\n$k$ = 500–6,500"),
          ("mid",  "Mid compression\n$k$ = 7,000–13,000"),
          ("low",  "Low compression\n$k$ = 13,500–19,500")]
FAMILIES = [("random", "Random"), ("degree-based", "Degree"),
            ("centrality-based", "Centrality"), ("ours(ablation)", "Ablation"),
            ("ours", "Ours")]
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"
BASE_COLOR = "#898781"
# sequential blue ramp (steps 250 -> 700) for lambda
# lam_cmap = LinearSegmentedColormap.from_list(
#     "lam", ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281", "#0d366b"])
# lam_norm = Normalize(0.0, 1.0)
lam_cmap = LinearSegmentedColormap.from_list(
    "lam_viridis", plt.cm.viridis(np.linspace(0.0, 0.8, 256)))
 
plt.rcParams.update({"font.size": 8, "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2,
                     "xtick.color": INK2, "ytick.color": INK2})
fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.3), sharey=True)
x_pos = {fam: i for i, (fam, _) in enumerate(FAMILIES)}
 
for ax, (lvl, title) in zip(axes, LEVELS):
    d = rank_df.filter(pl.col("compression_level") == lvl)
    others = d.filter(pl.col("family") != "ours")
    best_other = others.sort("avg_rank").row(0, named=True)
 
    # baselines: neutral gray, one column per family
    for fam, _ in FAMILIES[:-1]:
        sub = d.filter(pl.col("family") == fam)
        ax.scatter(np.full(sub.height, x_pos[fam]), sub["avg_rank"], s=30, color=BASE_COLOR,
                   edgecolor="white", linewidth=0.5, zorder=3, alpha=0.7)
 
    # ours: one point per lambda, colour = lambda
    ours = d.filter(pl.col("family") == "ours").with_columns(pl.col("file_type").cast(pl.Float64).alias("lam"))
    ax.scatter(np.full(ours.height, x_pos["ours"]), ours["avg_rank"], s=34,
               c=ours["lam"], cmap=lam_cmap, edgecolor="white", linewidth=0.8, zorder=4, alpha=0.7) # norm=lam_norm
    best = ours.sort("avg_rank").row(0, named=True)
    ax.scatter([x_pos["ours"]], [best["avg_rank"]], s=110, facecolor="none",
               edgecolor=INK, linewidth=1.0, zorder=5, alpha=0.7)
    ax.annotate(f"$\\lambda$ = {best['lam']:.1f}", (x_pos["ours"], best["avg_rank"]),
                xytext=(-9, 0), textcoords="offset points", ha="right", va="center",
                fontsize=7, color=INK)
 
    
        
 
    
 
    ax.set_title(title, fontsize=8, color=INK)
    ax.set_xticks(range(len(FAMILIES)))
    ax.set_xticklabels([lab for _, lab in FAMILIES], rotation=40, ha="right", fontsize=7)
    ax.set_xlim(-0.6, len(FAMILIES) - 0.05)
    ax.grid(axis="y", color=GRID, lw=0.5)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
 
axes[0].set_ylim(19.5, 0.5)                     # rank 1 on top
axes[0].set_yticks([1, 5, 10, 15, 19])
axes[0].set_ylabel("Average rank over five core metrics (1 = best)")
 
# sm = plt.cm.ScalarMappable(cmap=lam_cmap, norm=lam_norm)
sm = plt.cm.ScalarMappable(cmap=lam_cmap)
cbar = fig.colorbar(sm, ax=axes, fraction=0.025, pad=0.06)
cbar.set_label("Distance decay $\\lambda$ (ours)", color=INK2)
cbar.outline.set_visible(False)

fig.savefig("figures/rank_by_family.png", dpi=400, bbox_inches="tight")

# impact of K

In [ ]:
from matplotlib.ticker import FuncFormatter, MultipleLocator

# ---- series to plot: label, colour, line width, line style (order = legend order) ----
# colours: categorical slots 1-5 of a CVD-validated palette; references in gray
SERIES = {
    "0.9":                   ("Ours ($\\lambda{=}0.9$)",             "#2a78d6", 2.2, "-"),
    "personalized_pagerank": ("Pers. PageRank",                      "#1baf7a", 1.4, "-"),
    "highest_degree_dist_1": ("Highest degree ($d \\leq 1$)",        "#eda100", 1.4, "-"),
    "highest_degree":        ("Highest degree ($d \\leq D$)",        "#e87ba4", 1.4, "-"),
    "discrete_set_cover":    ("Set cover (ablation)",                "#898781", 1.3, (0, (5, 2))),
    "k_random_all_samples":  ("Random",                              "#898781", 1.3, (0, (1, 1.5))),
}
METRIC_TITLE = {
    "semantic_coverage":      "Semantic coverage ($\\uparrow$)",
    "distance_score":         "Distance score ($\\uparrow$)",
    "unique_rate":            "Unique rate ($\\uparrow$)",
    "unknown_rate":           "Unknown rate ($\\downarrow$)",
    "Tokens_per_concept":            "Tokens per concept ($\\downarrow$)",
    "uncovered_branche_rate": "Uncovered branch rate ($\\downarrow$)",
}
LEVEL_EDGES = [6750, 13250]                      # between high|mid and mid|low
LEVEL_LABELS = [(3500, "High"), (10000, "Mid"), (16500, "Low")]
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"


def plot_metrics_vs_k(df_k, metrics, ncols=3, panel_size=(2.5, 2), fname=None):
    """df_k: one row per (file_type, k) with the metric columns (random already averaged)."""
    plt.rcParams.update({"font.size": 8, "axes.edgecolor": "#c3c2b7",
                         "xtick.color": INK2, "ytick.color": INK2})
    n_panels = len(metrics) + 1                  # +1 slot for the legend
    nrows = -(-n_panels // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_size[0] * ncols, panel_size[1] * nrows),
                             sharex=True, constrained_layout=True)
    axes = axes.flatten()

    for ax, metric in zip(axes, metrics):
        # compression-level boundaries
        for x in LEVEL_EDGES:
            ax.axvline(x, color=GRID, lw=0.8, zorder=0)
        # references first (drawn underneath), ours last (on top)
        for ft, (label, color, lw, ls) in reversed(list(SERIES.items())):
            sub = df_k.filter(pl.col("file_type") == ft).sort("k")
            if sub.height == 0:
                continue
            ax.plot(sub["k"], sub[metric], color=color, lw=lw, ls=ls, label=label,
                    zorder=3 if ft == "0.9" else 2, solid_capstyle="round")
        ax.set_title(METRIC_TITLE.get(metric, metric), fontsize=8, color=INK, pad=11)
        ax.grid(axis="y", color=GRID, lw=0.5)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
        # compression-level labels just above the plot area
        for x, lab in LEVEL_LABELS:
            ax.text(x, 1.01, lab, transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                    fontsize=6, color=MUTED)

    for ax in axes[len(metrics):]:
        ax.axis("off")
    for ax in axes[:len(metrics)]:
        ax.xaxis.set_major_locator(MultipleLocator(5000))      # ticks at 0, 5k, 10k, 15k, 20k
        ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v/1000:.0f}k"))
        ax.tick_params(labelbottom=True)
    for ax in axes[(nrows - 1) * ncols:len(metrics)] if nrows > 1 else axes[:len(metrics)]:
        ax.set_xlabel("Token budget $k$", color=INK2)
    # bottom panels of incomplete last row: add xlabel to panels directly above empty slots
    for i in range(len(metrics), nrows * ncols):
        above = i - ncols
        if 0 <= above < len(metrics):
            axes[above].set_xlabel("Token budget $k$", color=INK2)

    handles, labels = axes[0].get_legend_handles_labels()
    order = [labels.index(SERIES[ft][0]) for ft in SERIES if SERIES[ft][0] in labels]
    axes[len(metrics)].legend([handles[i] for i in order], [labels[i] for i in order],
                              loc="center", frameon=False, fontsize=7.5, labelcolor=INK2,
                              handlelength=2.6)
    if fname:
        fig.savefig(fname, bbox_inches="tight", dpi=400)
    return fig

In [ ]:
METHODS = ["highest_degree_dist_1", "highest_degree", "closeness_centrality", "pagerank", "personalized_pagerank", "discrete_set_cover", "0.9"]
df_k = (perf_df_all
 .filter(pl.col("file_type").is_in(METHODS))
 .group_by(["family", "file_type", "k"])          # averages random's 20 draws
 .agg([pl.col(m).mean() for m in METRIC_CORE + ["uncovered_branche_rate"]])
 .sort(["family", "file_type", "k"])
 )

# main text: five core metrics
plot_metrics_vs_k(df_k, METRIC_CORE, fname="figures/metrics_vs_k.png")

df_k_all = (perf_df_all
 .filter(pl.col("file_type").is_in(METHODS + ["k_random_all_samples"]))
 .group_by(["family", "file_type", "k"])          # averages random's 20 draws
 .agg([pl.col(m).mean() for m in METRIC_CORE + ["uncovered_branche_rate"]])
 .sort(["family", "file_type", "k"])
 )

plot_metrics_vs_k(df_k, METRIC_CORE, fname="figures/metrics_vs_k.png")
plot_metrics_vs_k(df_k_all, METRIC_CORE, fname="figures/metrics_vs_k_all.png")

# impact on lambdas

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

METRIC_TITLE = {
    "semantic_coverage":      "Semantic coverage ($\\uparrow$)",
    "distance_score":         "Distance score ($\\uparrow$)",
    "unique_rate":            "Unique rate ($\\uparrow$)",
    "unknown_rate":           "Unknown rate ($\\downarrow$)",
    "Tokens_per_concept":     "Tokens per concept ($\\downarrow$)",
    "uncovered_branche_rate": "Uncovered branch rate ($\\downarrow$)",
}
# compression levels (same bins as the ranking section)
LEVELS = {"High": (500, 6500), "Mid": (7000, 13000), "Low": (13500, 19500)}
# ordered levels -> one blue ramp (light = high compression, dark = low) + marker as 2nd cue
LEVEL_STYLE = {"High": ("#6da7ec", "o"), "Mid": ("#2a78d6", "s"), "Low": ("#104281", "^")}
DEFAULT_LAMBDA = 0.9
INK, INK2, MUTED, GRID = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"


def lambda_level_means(perf_df_all, metrics, drop_zero=True):
    """Our method only: mean of each metric over the 13 budgets of each compression level.
    drop_zero: leave out lambda = 0 (reported in the table only)."""
    level = (pl.when(pl.col("k").is_between(*LEVELS["High"])).then(pl.lit("High"))
               .when(pl.col("k").is_between(*LEVELS["Mid"])).then(pl.lit("Mid"))
               .when(pl.col("k").is_between(*LEVELS["Low"])).then(pl.lit("Low")))
    df = (perf_df_all
          .filter(pl.col("category") == "greedy_tree_margin")
          .with_columns(pl.col("file_type").cast(pl.Float64).alias("lam"), level.alias("level"))
          .filter(pl.col("level").is_not_null()))
    if drop_zero:
        df = df.filter(pl.col("lam") > 0)
    return (df.group_by(["level", "lam"])
              .agg([pl.col(m).mean() for m in metrics])
              .sort(["level", "lam"]))


def plot_metrics_vs_lambda(df_lam, metrics, ncols=3, panel_size=(2.5, 2), fname=None):
    """df_lam: output of lambda_level_means (one row per level x lambda).
    Lines go through every lambda (incl. 0.91-0.99); markers only at steps of 0.1."""
    plt.rcParams.update({"font.size": 8, "axes.edgecolor": "#c3c2b7",
                         "xtick.color": INK2, "ytick.color": INK2})
    n_panels = len(metrics)                      # legend goes above the grid, no extra slot
    nrows = -(-n_panels // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(panel_size[0] * ncols, panel_size[1] * nrows),
                             sharex=True, constrained_layout=True)
    axes = axes.flatten()

    for ax, metric in zip(axes, metrics):
        # default lambda used in the other sections
        ax.axvline(DEFAULT_LAMBDA, color=GRID, lw=0.8, zorder=0)
        ax.text(DEFAULT_LAMBDA, 1.01, f"$\\lambda{{=}}{DEFAULT_LAMBDA}$", transform=ax.get_xaxis_transform(),
                ha="center", va="bottom", fontsize=6, color=MUTED)
        for lvl, (color, mk) in LEVEL_STYLE.items():
            sub = df_lam.filter(pl.col("level") == lvl).sort("lam")
            lam = sub["lam"].to_list()
            coarse = [i for i, l in enumerate(lam) if round(l * 100) % 10 == 0]  # 0.1, ..., 1.0
            ax.plot(sub["lam"], sub[metric], color=color, lw=1.6, marker=mk, markersize=2.5,
                     markeredgecolor="white", markeredgewidth=0.5,
                    label=f"{lvl} compression")
        ax.set_title(METRIC_TITLE.get(metric, metric), fontsize=8, color=INK, pad=11)
        ax.grid(axis="y", color=GRID, lw=0.5)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)

    for ax in axes[len(metrics):]:
        ax.axis("off")
    for ax in axes[:len(metrics)]:
        ax.xaxis.set_major_locator(MultipleLocator(0.2))
        ax.set_xlim(0.05, 1.03)                  # lambda = 0 is not plotted
        ax.tick_params(labelbottom=True)
    # x-label on the lowest panel of each column
    for col in range(ncols):
        idx = [i for i in range(col, len(metrics), ncols)]
        if idx:
            axes[idx[-1]].set_xlabel("Distance decay $\\lambda$", color=INK2)

    # legend above the panels (works on matplotlib < 3.7; bbox_inches="tight" keeps it in the file)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, 1.0),
               bbox_transform=fig.transFigure, ncol=3, frameon=False, fontsize=7.5,
               labelcolor=INK2, title="Mean over the 13 budgets of each level",
               title_fontsize=7, handlelength=2.6)
    if fname:
        fig.savefig(fname, bbox_inches="tight", dpi=400)
    return fig

In [ ]:
METRICS_LAMBDA = METRIC_CORE + ["uncovered_branche_rate"]
df_lam = lambda_level_means(perf_df_all, METRICS_LAMBDA)
plot_metrics_vs_lambda(df_lam, METRICS_LAMBDA, fname="figures/metrics_vs_lambda_no0.png")

In [ ]:
df_lam.write_csv("result_impact_by_lam.csv")

# correlation

In [ ]:
corr_cols = [
 'distance_score',
 'semantic_coverage',
 'uniqueness_entropy',
 'unique_rate',
 'uncovered_branche_rate',
 'unknown_rate',
 'tree_complexity',
 'Tokens_per_concept',   # renamed from conciseness in the setup cell
 ]
family_list = ["degree-based", "centrality-based", "random", "ours", "ours(ablation)"]
fig, ax = plt.subplots(5, 1, figsize=(10, 15), constrained_layout=True)

for family in family_list:
    corr_df = perf_df_all.filter(pl.col("family") == family).select(corr_cols).to_pandas().corr(method="spearman")

    im = ax[family_list.index(family)].imshow(corr_df.values, cmap="RdBu_r", vmin=-1, vmax=1)

    ax[family_list.index(family)].set_xticks(range(len(corr_cols)))
    ax[family_list.index(family)].set_yticks(range(len(corr_cols)))
    ax[family_list.index(family)].set_xticklabels([c.replace("_", " ") for c in corr_cols], rotation=45, ha="right", fontsize=8)
    ax[family_list.index(family)].set_yticklabels([c.replace("_", " ") for c in corr_cols], fontsize=8)
    ax[family_list.index(family)].set_title(family, fontsize=10)

    for i in range(len(corr_cols)):
        for j in range(len(corr_cols)):
            v = corr_df.values[i, j]
            ax[family_list.index(family)].text(j, i, f"{v:.2f}", ha="center", va="center",
                    color="white" if abs(v) > 0.6 else "black", fontsize=7)

fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson correlation")
fig.suptitle("Correlation matrix: metrics vs k", fontsize=12)
# fig.tight_layout()

corr_df